### Sprints: Bronze to Silver
Clean and transform raw sprints data from `formula1_incr.bronze.sprints` into `formula1_incr.silver.sprints`.

#### Setup
- `01.environment-config` → loads catalog name, bronze/silver schema names
- `03.silver_helpers` → loads the `write_to_silver()` function we use to save data

In [0]:
%run ../00-common/01.environment-config 

In [0]:
%run ../00-common/03.silver_helpers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.sprints'
silver_table = f'{catalog_name}.{silver_schema}.sprints'

#### Read Bronze
- Read raw data from the bronze table filtered by `batch_id`

In [0]:
# Read Bronze table
sprints_df = spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)

#### Drop Columns
- Remove `url` column — not needed for analysis

In [0]:
# Drop unnecessary columns
sprints_drop_df = sprints_df.drop('url')
display(sprints_drop_df)

#### Rename Columns
- Convert camelCase to snake_case for consistency

In [0]:
# Rename columns to snake_case
sprints_renamed_df = (sprints_drop_df
    .withColumnsRenamed({'constructorId': 'constructor_id',
                         'driverId': 'driver_id',
                         'raceName': 'race_name',
                         'date': 'race_date',
                         'grid' : 'grid_position',
                         'laps':'Completed_laps',
                         'number' : 'car_number',
                         'position':'final_position',
                         'positionText':'final_position_text'})
)

#### Remove Nulls
- Drop rows where key columns (`season`, `round`, `constructor_id`, `driver_id`) are null

In [0]:
# Remove rows with null key values
sprints_valid_df = (
   sprints_renamed_df
    .filter(
        col('season').isNotNull() &
        col('round').isNotNull() &
        col('constructor_id').isNotNull() &
        col('driver_id').isNotNull()
    )
)


#### Remove Duplicates
- Keep one row per driver per race using `dropDuplicates()`

In [0]:
# Remove duplicates
sprints_distinct_df = sprints_valid_df.dropDuplicates(['driver_id','season','constructor_id','round'])
display(sprints_distinct_df)

#### Title Case
- Apply `initcap()` to `race_name` so it looks clean

In [0]:
# Apply title case to race_name
sprints_final_df = sprints_distinct_df.withColumn('race_name', initcap(col('race_name')))

#### Write to Silver
- If the silver table doesn't exist yet, it creates it from scratch
- If it already exists, it merges new/updated rows using `write_to_silver()` (insert new, update changed)

In [0]:
write_to_silver(
    input_df=sprints_final_df,
    target_table=silver_table,
    merge_condition="""
    s.season = t.season 
    AND s.round = t.round 
    AND s.constructor_id = t.constructor_id 
    AND s.driver_id = t.driver_id""",
    columns_to_update=[
        "race_date",
        "race_name",
        "round",
        "season",
        "constructor_id",
        "driver_id",
        "grid_position",
        "Completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

#### Verify
- Read back the silver table to confirm data was saved

In [0]:
# Verify
spark.read.table(silver_table).display()